# HS4002 Week 11

## Panel Data Models

Panel data follows the **same units** (people, players, countries) across multiple time points, allowing us to control for unobserved unit-level heterogeneity.

In Python, the `linearmodels` library provides panel regression — equivalent to R's `plm` package.

We'll use NBA player data (2013–2017) to ask: **how does a player's age affect their scoring performance?**

In [3]:
%pip install pandas numpy plotnine linearmodels stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

  Using cached linearmodels-7.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached pyhdfe-0.2.1-py3-none-any.whl.metadata (4.5 kB)
Using cached linearmodels-7.1-cp312-cp312-macosx_11_0_arm64.whl (2.1 MB)
Using cached pyhdfe-0.2.1-py3-none-any.whl (20 kB)

[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup: Load and Prepare the NBA Dataset

In [ ]:
# Data source: https://www.kaggle.com/datasets/drgilermo/nba-players-stats
# NBA player statistics 1950-2017 from basketball-reference.com
raw_df = pd.read_csv("bball.csv").iloc[:, 1:]  # drop the unnamed index column


# all of these are measures for the performance of a player
df = (
	raw_df[raw_df["Year"] >= 2013]
	.groupby(["Player", "Year"])
	.agg(
		games_played    = ("G",    "sum"),
		games_started   = ("GS",   "sum"),
		minutes_played  = ("MP",   "sum"),
		age             = ("Age",  "mean"),
		personal_fouls  = ("PF",   "sum"),
		points          = ("PTS",  "sum"),
		true_shooting   = ("TS%",  "mean"),
		assists         = ("AST",  "sum"),
		steals          = ("STL",  "sum"),
		blocks          = ("BLK",  "sum"),
		fga             = ("FGA",  "sum"),
		fg              = ("FG",   "sum"),
		threeP          = ("3P",   "sum"),
		threePA         = ("3PA",  "sum"),
		twoP            = ("2P",   "sum"),
		twoPA           = ("2PA",  "sum"),
		ft              = ("FT",   "sum"),
		fta             = ("FTA",  "sum"),
		def_rb          = ("DRB",  "sum"),
		off_rb          = ("ORB",  "sum"),
	)
	.assign(ppg=lambda x: x["points"] / x["games_played"]) # ppg is the points scored per game
	.reset_index()
)
print(df.shape)
df.head()

(2403, 23)


,Player,Year,games_played,games_started,minutes_played,age,personal_fouls,points,true_shooting,assists,...,fg,threeP,threePA,twoP,twoPA,ft,fta,def_rb,off_rb,ppg
0,A.J. Hammons,2017.0,22.0,0.0,163.0,24.0,21.0,48.0,0.472000,4.0,...,17.0,5.0,10.0,12.0,32.0,9.0,20.0,28.0,8.0,2.181818
1,A.J. Price,2013.0,57.0,22.0,1278.0,26.0,73.0,441.0,0.501000,205.0,...,161.0,70.0,200.0,91.0,213.0,49.0,62.0,94.0,20.0,7.736842
2,A.J. Price,2014.0,28.0,0.0,99.0,27.0,5.0,44.0,0.469000,13.0,...,19.0,6.0,22.0,13.0,24.0,0.0,2.0,9.0,1.0,1.571429
3,A.J. Price,2015.0,52.0,0.0,648.0,28.0,30.0,266.0,0.376750,92.0,...,102.0,30.0,114.0,72.0,160.0,32.0,48.0,52.0,12.0,5.115385
4,Aaron Brooks,2013.0,106.0,40.0,1994.0,28.0,188.0,752.0,0.500667,228.0,...,286.0,100.0,268.0,186.0,364.0,80.0,104.0,134.0,26.0,7.094340


## Setting up the Panel Index

`linearmodels` requires a **MultiIndex** of (entity, time). Equivalent to the `index=c('Player','Year')` argument in R's `plm()`.

In [ ]:
# Set the panel index — required by linearmodels
# tells the computer which is the entity (i.e. player) and which one is the time (i.e. Year)
df_panel = df.set_index(['Player', 'Year'])

# A Tutorial for Panel Models

## Pooled OLS

Equivalent to a regular `lm()` in R — ignores the panel structure.

In [ ]:
# we assume that all the people are randomly chosen when we use OLS, but that is not true
pool_model = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=df_panel)
pool_result = pool_model.fit()
print(pool_result.summary)

# with one unit of age, you score 0.6 more points
# both are statistically significant
# there is a curved relationship between age and ppg -- your performance increases with age, until you reach your prime! your performance then declines

                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.6757
Estimator:                  PooledOLS   R-squared (Between):              0.6103
No. Observations:                2403   R-squared (Within):               0.0318
Date:                Wed, Oct 28 2026   R-squared (Overall):              0.6757
Time:                        12:07:24   Log-likelihood                   -7587.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2501.2
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             2501.2
                            

## Fixed Effects (Within) Model

Adds `EntityEffects` to control for time-invariant player characteristics. Equivalent to `plm(..., model='within')` in R.

In [ ]:
fe_model = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=df_panel)
fe_result = fe_model.fit()
print(fe_result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.1774
Estimator:                   PanelOLS   R-squared (Between):             -66.812
No. Observations:                2403   R-squared (Within):               0.1774
Date:                Wed, Oct 28 2026   R-squared (Overall):             -43.473
Time:                        12:12:04   Log-likelihood                   -5167.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      170.80
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,1584)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             170.80
                            

Interpret the coefficients. How does the fixed-effects estimate differ from the pooled OLS estimate?

In [ ]:
# with one unit of age, you score 5.6 more points -- way higher than the OLS model predicted!
# this is because you only compare a player against himself, and not with other players
# still has a quadratic relationship

## Random Effects Model

Equivalent to `plm(..., model='random')` in R.

In [ ]:
# random effect uses a little infomation from other players to predict how a single player performs
# assumes that each player is not that unique. you need to justify why you use that model!

re_model = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=df_panel)
re_result = re_model.fit()
print(re_result.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.3804
Estimator:              RandomEffects   R-squared (Between):              0.6371
No. Observations:                2403   R-squared (Within):               0.0412
Date:                Wed, Oct 28 2026   R-squared (Overall):              0.6573
Time:                        12:12:37   Log-likelihood                   -5810.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      737.02
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             737.02
                            

## Model Diagnostics

### Lagrange Multiplier Test

Tests whether individual effects are present (i.e., whether FE/RE is needed over pooled OLS). Equivalent to R's `plmtest(fe_model, effect='individual')`.

In [ ]:
# Breusch-Pagan LM test for random effects
# Under H0 (no individual effects), we prefer pooled OLS.
# Equivalent to R's plmtest(..., effect='individual', type='bp').
from scipy.stats import chi2

resid = pool_result.resids
e_i   = resid.groupby(level=0).sum()
ssr   = float((resid ** 2).sum())
A1    = float((e_i ** 2).sum()) / ssr - 1
T_i   = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects:')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

# since p-value = 0, we reject the null hypothesis
# there are fixed effects in the model

LM test for individual effects:
LM statistic = 2381.2182
df = 1
p-value = 0.0000


### Hausman Test

Tests whether RE estimates are consistent (i.e., whether FE or RE is preferred). Under H0, RE is consistent and more efficient. Equivalent to R's `phtest(fe_model, re_model)`.

In [ ]:
# Hausman test: compare FE and RE estimates
# Under H0 (RE consistent), the difference in coefficients should be small
from scipy.stats import chi2

common_params = fe_result.params.index.intersection(re_result.params.index)

beta_diff = fe_result.params[common_params] - re_result.params[common_params]
cov_diff  = (
	fe_result.cov.loc[common_params, common_params] -
	re_result.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val  = chi2.sf(h_stat, df=len(common_params))
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)

# since p-value is small, we reject the null hypothesis. the fixed effects model is preferred

Hausman statistic = 285.0094
df = 2
p-value = 0.0000
Prefer FE over RE: True


## Comparing the Models

As in earlier weeks, `stargazer` collects the fitted models into one regression table. Reading pooled OLS, fixed effects, and random effects side by side is easier than scrolling through three summaries. Equivalent to R's `stargazer(lm1, fe_glm1, re_glm1)`.

In [ ]:
from stargazer.stargazer import Stargazer

sg = Stargazer([pool_result, fe_result, re_result])
sg.title('Points per Game and Age')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Points per game')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg.covariate_order(['age', 'I(age ** 2)'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
})

# Ending the cell on the object itself renders the table in the notebook
sg

# results are statistically signficiant. but within effects might be too small

# Independent Analysis: Has Sports Science Extended Career Peaks?

Compare players from 2008–2012 with players from 2013–2017. Does age affect scoring differently across the two eras?

In [ ]:
# (1) create two data sets, one for each era.

df = (
	raw_df[raw_df["Year"] < 2013]
	.groupby(["Player", "Year"])
	.agg(
		games_played    = ("G",    "sum"),
		games_started   = ("GS",   "sum"),
		minutes_played  = ("MP",   "sum"),
		age             = ("Age",  "mean"),
		personal_fouls  = ("PF",   "sum"),
		points          = ("PTS",  "sum"),
		true_shooting   = ("TS%",  "mean"),
		assists         = ("AST",  "sum"),
		steals          = ("STL",  "sum"),
		blocks          = ("BLK",  "sum"),
		fga             = ("FGA",  "sum"),
		fg              = ("FG",   "sum"),
		threeP          = ("3P",   "sum"),
		threePA         = ("3PA",  "sum"),
		twoP            = ("2P",   "sum"),
		twoPA           = ("2PA",  "sum"),
		ft              = ("FT",   "sum"),
		fta             = ("FTA",  "sum"),
		def_rb          = ("DRB",  "sum"),
		off_rb          = ("ORB",  "sum"),
	)
	.assign(ppg=lambda x: x["points"] / x["games_played"]) # ppg is the points scored per game
	.reset_index()
)
print(df.shape)
df.head()

In [ ]:
# (2) fit models for each of the two data sets

In [ ]:
# (3) run diagnostics, compare, and interpret for 2008-2012.

In [ ]:
# (4) run diagnostics, compare, and interpret for 2013-2017.

In [ ]:
# (5) what differences do you find in the age-scoring relationship across the two eras?